## KG Construction v2 — Constrained Prompt

### Why this notebook exists

This notebook builds a second version of the Sexism Knowledge Graph (`sexism_kg_v2.json`) using an improved extraction prompt. It exists alongside `04_kg_construction.ipynb` (which produced `sexism_kg.json`) to enable a direct comparison between two KG construction approaches as part of the research methodology.

### What was wrong with KG v1

The original KG was built using an unconstrained prompt — Gemini was shown the post's EDOS category as a hint but was free to choose any relation it considered appropriate based on the post text alone. This led to significant misalignment between the post's ground truth category and the relation Gemini extracted:

| Relation | Expected Category | Alignment (v1) |
|---|---|---|
| THREATENED_WITH | threats | 70.6% |
| EXPRESSED_ANIMOSITY_TOWARDS | animosity | 70.8% |
| FRAMED_AS_INFERIOR | derogation | 58.5% |
| STEREOTYPED_AS | derogation | 55.6% |
| IDEOLOGICALLY_DISCREDITED | prejudiced discussions | 52.0% |
| ASSIGNED_TO_ROLE | prejudiced discussions | 25.3% |

ASSIGNED_TO_ROLE was the worst offender — only 25.3% of triples using this relation came from prejudiced discussion posts. It was extracted from animosity, derogation, and threat posts equally, making it effectively noise. It was removed from `sexism_kg_clean.json` as a post-processing step.

### What changed in v2

**1. Category-constrained prompt**
The new prompt explicitly forces Gemini to use only the relation(s) that align with the post's EDOS category:
- derogation posts → STEREOTYPED_AS or FRAMED_AS_INFERIOR only
- animosity posts → EXPRESSED_ANIMOSITY_TOWARDS only
- threat posts → THREATENED_WITH only
- prejudiced discussion posts → ASSIGNED_TO_ROLE or IDEOLOGICALLY_DISCREDITED only

This directly fixes the alignment problem at construction time rather than requiring post-processing.

**2. ASSIGNED_TO_ROLE restored**
ASSIGNED_TO_ROLE was removed from `sexism_kg_clean.json` because of poor alignment under the old unconstrained prompt. With the new constrained prompt it is restricted to prejudiced discussion posts only — the category it was designed to represent. It is kept in v2 to maximise coverage for the smallest EDOS category (333 training posts).

**3. Canonical entity form enforced in prompt**
The new prompt explicitly instructs Gemini to use `women` instead of `she/her`, `men` instead of `he/him`, and `feminists` instead of `they/them`. This eliminates the need for post-processing normalization.

### Expected outcome

Alignment percentages should improve significantly — target above 85% for all relations. The constrained prompt trades some extraction flexibility for much higher semantic precision, which is more valuable for the downstream KG-guided LLM reasoning task.

### Files produced

- `kg/sexism_kg_v2.json` — new KG with constrained extraction
- `kg/kg_v2_checkpoint.json` — construction checkpoint (resumable)

### Relationship to other notebooks

- `04_kg_construction.ipynb` → produces `sexism_kg.json` (v1, unconstrained)
- `04_kg_construction.ipynb` Cell 15 → produces `sexism_kg_clean.json` (v1 cleaned)
- **This notebook** → produces `sexism_kg_v2.json` (v2, constrained)
- `05_proposed_model_A.ipynb` → uses `sexism_kg_clean.json`
- `06_proposed_model_BC.ipynb` → uses `sexism_kg_v2.json` + semantic retrieval

In [1]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import json
import time
import pandas as pd
from google import genai
from dotenv import load_dotenv
from collections import defaultdict
from data_loader import load_edos_data
DATA_DIR='../data/'
KG_DIR='../kg/'
RESULTS_DIR='../results/'
os.makedirs(KG_DIR, exist_ok=True)
load_dotenv('../.env')
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError('GEMINI_API_KEY not found in .env file')
client=genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL='gemini-2.5-flash-lite'
print('Setup complete.')

c:\Users\Ashwin Nair\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete.


In [2]:
time.sleep(3)
## Testing GEMINI
response = client.models.generate_content(model=GEMINI_MODEL, contents='Reply with OK only.')
print(f'Gemini API working: {response.text.strip()}')

Gemini API working: OK


In [3]:
train_df, _, _ = load_edos_data(DATA_DIR, task='B') ##
print(f'Sexist training posts for KG construction: {len(train_df)}') 
print(f'\nCategory distribution:') 
print(train_df['label'].value_counts().to_string())

Sexist training posts for KG construction: 3398

Category distribution:
label
2. derogation                               1590
3. animosity                                1165
4. prejudiced discussions                    333
1. threats, plans to harm and incitement     310


In [4]:
# All 6 relations kept — ASSIGNED_TO_ROLE was performing badly due to
# the old unconstrained prompt, not because the relation itself is wrong.
# The new constrained prompt fixes alignment by forcing category-specific relations.

KG_SCHEMA = {
    'STEREOTYPED_AS':'2. derogation',
    'FRAMED_AS_INFERIOR':'2. derogation',
    'ASSIGNED_TO_ROLE':'4. prejudiced discussions',
    'THREATENED_WITH':'1. threats, plans to harm and incitement',
    'EXPRESSED_ANIMOSITY_TOWARDS':'3. animosity',
    'IDEOLOGICALLY_DISCREDITED':'4. prejudiced discussions',
}

VALID_RELATIONS = set(KG_SCHEMA.keys())

print('KG Relation Schema (v2 — all 6 relations):')
for relation, category in KG_SCHEMA.items():
    print(f'  {relation} -> {category}')

KG Relation Schema (v2 — all 6 relations):
  STEREOTYPED_AS -> 2. derogation
  FRAMED_AS_INFERIOR -> 2. derogation
  ASSIGNED_TO_ROLE -> 4. prejudiced discussions
  THREATENED_WITH -> 1. threats, plans to harm and incitement
  EXPRESSED_ANIMOSITY_TOWARDS -> 3. animosity
  IDEOLOGICALLY_DISCREDITED -> 4. prejudiced discussions


In [5]:
EXTRACTION_PROMPT = """You are a knowledge graph construction expert specialising in sexist language analysis.

Extract semantic triples from the following social media post. Each triple must follow the format:
(subject, RELATION, object)

The post belongs to this category: {category}

Based on the category, you MUST only use these relations:
- If category is "1. threats, plans to harm and incitement" → use ONLY: THREATENED_WITH
- If category is "2. derogation" → use ONLY: STEREOTYPED_AS or FRAMED_AS_INFERIOR
- If category is "3. animosity" → use ONLY: EXPRESSED_ANIMOSITY_TOWARDS
- If category is "4. prejudiced discussions" → use ONLY: ASSIGNED_TO_ROLE or IDEOLOGICALLY_DISCREDITED

Relation definitions:
- STEREOTYPED_AS: subject is portrayed through a negative stereotype
- FRAMED_AS_INFERIOR: subject is portrayed as less capable or less worthy
- ASSIGNED_TO_ROLE: subject is assigned a specific gender role or domestic expectation
- THREATENED_WITH: subject is threatened with harm or violence
- EXPRESSED_ANIMOSITY_TOWARDS: subject is the target of hostility or hatred
- IDEOLOGICALLY_DISCREDITED: subject's position or role is ideologically dismissed

Post: "{text}"
Category: {category}

Rules:
- Extract 1 to 3 triples maximum
- The subject should be the entity being targeted (usually women, feminists, girls)
- The object should be the trait, role, threat, or characteristic being assigned to them
- Subject and object should be short noun phrases (2-5 words)
- Use canonical form: women (not she/her), men (not he/him), feminists (not they/them)
- Only extract triples clearly supported by the post
- If no clear triple exists, return NONE

Respond ONLY in this exact format (one triple per line):
(subject, RELATION, object)

Or if no triple exists:
NONE"""

In [6]:
def parse_triples(response_text: str, valid_relations: set) -> list: ## Parse Gemini response into list of triple dicts
    triples=[]
    lines=response_text.strip().split('\n')

    for line in lines:
        line=line.strip() 
        if not line or line.upper()=='NONE':
            continue
        if line.startswith('(') and line.endswith(')'):
            line=line[1:-1]
        parts=[p.strip() for p in line.split(',')] 
        if len(parts)!=3:
            continue
        subject, relation, obj=parts
        relation=relation.upper().strip()

        if relation not in valid_relations: 
            matched=False
            for valid_rel in valid_relations:
                if valid_rel in relation or relation in valid_rel:
                    relation=valid_rel
                    matched=True
                    break
            if not matched:
                continue

        triples.append({'subject':subject.lower().strip(),'relation':relation,'object':obj.lower().strip()})

    return triples

In [ ]:
def extract_triples(text: str, category: str, max_retries: int = 3) -> list: ##Call Gemini to extract KG triples from a single post.
    prompt=EXTRACTION_PROMPT.format(text=text, category=category) 

    for attempt in range(max_retries): 
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
            if response is None or not hasattr(response, 'text') or response.text is None:
                raise ValueError('Empty response from Gemini')

            raw=response.text.strip()
            triples=parse_triples(raw, VALID_RELATIONS)
            return triples

        except Exception as e:
            error_str = str(e)
            if '429' in error_str or '503' in error_str: 
                wait=60 * (attempt + 1)
                print(f'Server busy (attempt {attempt+1}/{max_retries}) - waiting {wait}s')
                time.sleep(wait)
            elif attempt<max_retries-1:
                print(f'Attempt{attempt+1} failed: {e} - retrying in 15s')
                time.sleep(15)
            else:
                print(f'Failed after {max_retries} attempts: {e}')

    return []

In [9]:
print('Testing constrained extraction on 5 posts:\n')
# Test one post per category to verify constraint is working
for category in train_df['label'].unique():
    sample=train_df[train_df['label']==category].sample(1, random_state=42)
    row=sample.iloc[0]
    print(f'Post:{row["text"]}')
    print(f'Category: {row["label"]}')
    triples=extract_triples(row['text'], row['label'])
    if triples:
        for t in triples:
            print(f'Triple:({t["subject"]}, {t["relation"]}, {t["object"]})')
    else:
        print(' No triples extracted')
    print()
    time.sleep(3)

Testing constrained extraction on 5 posts:

Post:I had a girlfriend who went on and on about how women were so much better at multitasking. In the end I had to say "if you're so good at multitasking then how come you can't sit down and shut up"
Category: 3. animosity
Triple:(women, EXPRESSED_ANIMOSITY_TOWARDS, talking too much)
Triple:(women, EXPRESSED_ANIMOSITY_TOWARDS, inability to be quiet)

Post:Didn't they used to burn these bitches... er witches.
Category: 2. derogation
Triple:("these bitches", STEREOTYPED_AS, "witches")

Post:I probably wouldn't brag about that because usually when women make those choices they end up either being single moms or knowing the domestic violence hotline number extremely well.
Category: 4. prejudiced discussions
Triple:(women, ASSIGNED_TO_ROLE, single moms)
Triple:(women, ASSIGNED_TO_ROLE, knowing domestic violence hotline)

Post:lock her ass up
Category: 1. threats, plans to harm and incitement
Triple:(women, THREATENED_WITH, incarceration)

